In [ ]:
import os
import subprocess
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
import requests

In [ ]:
url = "https://files.rcsb.org/download/1VPF.pdb"
response = requests.get(url)
with open("vefg.pdb", "wb") as f:
    f.write(f'../data/external/{response.content}')

In [ ]:
# === Configurations ===
vina_path = "../references/vina_1.2.7_win.exe"  # Assume vina is in PATH
receptor_pdb = "../data/external/vefg.pdb"  # VEGF structure in PDB format
output_dir = "../data/processed/docking_results"
peptides = pd.read_csv('../data/processed/sequences_reviewed.csv', index_col=0)

In [ ]:
# Box center and size for VEGF binding site (adjust manually)
center = (10, 10, 10)
size = (20, 20, 20)



In [ ]:
os.makedirs(output_dir, exist_ok=True)

def save_peptide_pdb(peptide_seq, filename):
    """Save a peptide FASTA and convert it to PDB using PyMOL or other tools"""
    fasta = f">pep\n{peptide_seq}"
    with open("tmp_peptide.fasta", "w") as f:
        f.write(fasta)
    
    # Use external tool for peptide PDB generation. Example with PyMOL (optional)
    pymol_script = f"""
    seq = '{peptide_seq}'
    cmd.fragment('peptide', origin=1)
    cmd.save('{filename}')
    cmd.quit()
    """
    with open("build_peptide.pml", "w") as f:
        f.write(pymol_script)
    subprocess.run(["pymol", "-c", "build_peptide.pml"])

In [ ]:
def prepare_pdbqt(input_pdb, output_pdbqt, is_receptor=False):
    """Convert PDB to PDBQT using Open Babel"""
    if is_receptor:
        cmd = ["obabel", input_pdb, "-O", output_pdbqt, "--gen3d"]
    else:
        cmd = ["obabel", input_pdb, "-O", output_pdbqt, "--gen3d", "--partialcharge", "gasteiger"]
    subprocess.run(cmd, check=True)


In [ ]:
def dock_ligand(receptor_pdbqt, ligand_pdbqt, output_pdbqt, log_file):
    """Run AutoDock Vina"""
    cmd = [
        vina_path,
        "--receptor", receptor_pdbqt,
        "--ligand", ligand_pdbqt,
        "--center_x", str(center[0]),
        "--center_y", str(center[1]),
        "--center_z", str(center[2]),
        "--size_x", str(size[0]),
        "--size_y", str(size[1]),
        "--size_z", str(size[2]),
        "--out", output_pdbqt,
        "--log", log_file
    ]
    subprocess.run(cmd, check=True)

## Main workflow

In [ ]:
# 1. Prepare receptor
receptor_pdbqt = os.path.join(output_dir, "receptor.pdbqt")
prepare_pdbqt(receptor_pdb, receptor_pdbqt, is_receptor=True)

# 2. Process each peptide
for i, pep in enumerate(peptides):
    print(f"Processing peptide: {pep}")
    pep_pdb = os.path.join(output_dir, f"pep_{i}.pdb")
    pep_pdbqt = os.path.join(output_dir, f"pep_{i}.pdbqt")
    out_pdbqt = os.path.join(output_dir, f"out_{i}.pdbqt")
    log_file = os.path.join(output_dir, f"log_{i}.txt")

    save_peptide_pdb(pep, pep_pdb)
    prepare_pdbqt(pep_pdb, pep_pdbqt)
    dock_ligand(receptor_pdbqt, pep_pdbqt, out_pdbqt, log_file)

print("Docking complete.")